# Two Ways from Tidy to Wide

DS 2023 | Communicating with Data

## Overview

This notebook introduces a recipe for create a wide table from a tidy table.

It assumens the following:

- You have a tidy DataFrame $D$ with two grouping variables $g_1$ and $g_2$ and one measurement variable $m$.

- You want to create a wide table $W$ with $g_1$ on axis 0, $g_2$ on axis 1, and $m$ data in the cells.

It describes two method for doing so under two conditions.

- **Conditions**: Do the two grouping variables combine to produce a unique set over the observations or not?

- **Methods**: Using pivoting or using unstacking.


## Flowchart

This flowchart summarizes the pattern.

```mermaid
flowchart TD
    
    C["Check if g1 x g2 combinations are unique over D by using D.value_counts(['g1','g2'])"]
    
    C --> D{"Does every count == 1?"}

    D -->|"Yes: each pair occurs<br>at most once"| E["Pure reshape<br>no aggregation needed"]
    
    D -->|"No: some pairs repeat"| F["Need to reshape and collapse with aggregation, e.g. mean"]

    E --> E1["D.pivot(index='g1',<br>columns='g2',<br>values='m')"]
    
    E --> E2["D.set_index(['g1','g2'])['m']<br>.unstack('g2')"]

    F --> F1["D.pivot_table(index='g1',<br>columns='g2',<br>values='m',<br>aggfunc='mean')"]
    
    F --> F2["D.groupby(['g1','g2'])['m']<br>.mean()['g2'].unstack()"]

    E1 --> Z["W: rows = G1 levels, cols = G2 levels, cells = M"]
    
    E2 --> Z
    F1 --> Z
    F2 --> Z


    classDef unique fill:#d8f0d8,stroke:#4a8a4a,color:#1c3a1c
    classDef dup fill:#fde4d4,stroke:#c07a45,color:#4a2a12
    classDef err fill:#f8d7da,stroke:#a94442,color:#5a1d20
    class E,E1,E2 unique
    class F,F1,F2 dup
    class W err
```

## Example 1

### Get the data

In [3]:
import pandas as pd
import seaborn as sns

healthexp = sns.load_dataset("healthexp")
healthexp.head()

,Year,Country,Spending_USD,Life_Expectancy
0,1970,Germany,252.311,70.6
1,1970,France,192.143,72.2
2,1970,Great Britain,123.993,71.9
3,1970,Japan,150.437,72.0
4,1970,USA,326.961,70.9


### Choose the variables

In [19]:
a_g1 = 'Year'
a_g2 = 'Country'
a_m = 'Spending_USD'

### Check for uniqueness

Value counts returns results in descending order, so any values greater than 1 would appear first.

In [69]:
healthexp.value_counts([a_g1,a_g2]).head(1)

Year  Country
1970  Germany    1
Name: count, dtype: int64

No values greater than 1, so unique.

### Reshape

#### Method 1: Pivoting

In [22]:
healthexp.pivot(index=a_g1, columns=a_g2, values=a_m).head(10)

Country,Canada,France,Germany,Great Britain,Japan,USA
Year,,,,,,
1970,NaN,192.143,252.311,123.993,150.437,326.961
1971,313.391,NaN,298.251,134.172,163.854,357.988
1972,NaN,NaN,337.364,NaN,185.390,397.097
1973,NaN,NaN,384.541,NaN,205.778,439.302
1974,NaN,NaN,452.744,NaN,242.018,495.114
1975,NaN,363.610,532.481,NaN,284.269,560.750
1976,543.337,NaN,591.098,NaN,303.725,638.851
1977,NaN,NaN,647.352,NaN,340.628,726.241
1978,NaN,NaN,729.457,NaN,392.577,808.884


#### Method 2: Unstacking

In [18]:
healthexp.set_index([a_g1, a_g2])[a_m].unstack().head(10)

Country,Canada,France,Germany,Great Britain,Japan,USA
Year,,,,,,
1970,NaN,192.143,252.311,123.993,150.437,326.961
1971,313.391,NaN,298.251,134.172,163.854,357.988
1972,NaN,NaN,337.364,NaN,185.390,397.097
1973,NaN,NaN,384.541,NaN,205.778,439.302
1974,NaN,NaN,452.744,NaN,242.018,495.114
1975,NaN,363.610,532.481,NaN,284.269,560.750
1976,543.337,NaN,591.098,NaN,303.725,638.851
1977,NaN,NaN,647.352,NaN,340.628,726.241
1978,NaN,NaN,729.457,NaN,392.577,808.884


## Example 2

### Get the data

In [67]:
mpg = sns.load_dataset('mpg')
mpg.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


### Choose the variables

In [24]:
b_g1 = 'model_year'
b_g2 = 'origin'
b_m = 'mpg'

### Check for uniqueness

Value counts returns results in descending order, so any values greater than 1 would appear first.

In [68]:
mpg.value_counts([b_g1,b_g2]).head(1)

model_year  origin
73          usa       29
Name: count, dtype: int64

Not unique.

### Aggregate and Reshape

#### Method 1: Pivoting with aggregation

In [70]:
mpg.pivot_table(index=b_g1, columns=b_g2, values=b_m, aggfunc='mean')

origin,europe,japan,usa
model_year,,,
70,25.200000,25.500000,15.272727
71,28.750000,29.500000,18.100000
72,22.000000,24.200000,16.277778
73,24.000000,20.000000,15.034483
74,27.000000,29.333333,18.333333
75,24.500000,27.500000,17.550000
76,24.250000,28.000000,19.431818
77,29.250000,27.416667,20.722222
78,24.950000,29.687500,21.772727


#### Method 2: Aggregating then unstacking

In [71]:
mpg.groupby([b_g1, b_g2])[b_m].mean().unstack()

origin,europe,japan,usa
model_year,,,
70,25.200000,25.500000,15.272727
71,28.750000,29.500000,18.100000
72,22.000000,24.200000,16.277778
73,24.000000,20.000000,15.034483
74,27.000000,29.333333,18.333333
75,24.500000,27.500000,17.550000
76,24.250000,28.000000,19.431818
77,29.250000,27.416667,20.722222
78,24.950000,29.687500,21.772727


## General Function

We define a general function that takes a tidy dataframe as its first argument and returns a wide dataframe.

In [72]:
def tidy2wide(tidy, g1, g2, m, method='pivot', aggfunc='mean'):

    unique = True if tidy.value_counts([g1,g2]).head(1).values[0] == 1 else False 

    if unique:
        print("Group combos are unique.")
        if method == 'pivot':
            wide = tidy.pivot(index=g1, columns=g2, values=m)
        elif method == 'unstack':
            wide = tidy.set_index([g1,g2])[m].unstack()
        else:
            raise ValueError("Invalid method.")
            
    else:
        print("Group combos are not unique.")
        if method == 'pivot':
            wide = tidy.pivot_table(index=g1, columns=g2, values=m, aggfunc=aggfunc)
        elif method == 'unstack':
            wide = tidy.groupby([g1,g2])[m].mean().unstack()
        else:
            raise ValueError("Invalid method and/or aggfunc.")

    return wide

In [73]:
tidy2wide(mpg, 'model_year', 'origin', 'mpg')

Group combos are not unique.


origin,europe,japan,usa
model_year,,,
70,25.200000,25.500000,15.272727
71,28.750000,29.500000,18.100000
72,22.000000,24.200000,16.277778
73,24.000000,20.000000,15.034483
74,27.000000,29.333333,18.333333
75,24.500000,27.500000,17.550000
76,24.250000,28.000000,19.431818
77,29.250000,27.416667,20.722222
78,24.950000,29.687500,21.772727


In [74]:
tidy2wide(mpg, 'model_year', 'origin', 'mpg', method='unstack')

Group combos are not unique.


origin,europe,japan,usa
model_year,,,
70,25.200000,25.500000,15.272727
71,28.750000,29.500000,18.100000
72,22.000000,24.200000,16.277778
73,24.000000,20.000000,15.034483
74,27.000000,29.333333,18.333333
75,24.500000,27.500000,17.550000
76,24.250000,28.000000,19.431818
77,29.250000,27.416667,20.722222
78,24.950000,29.687500,21.772727


In [75]:
tidy2wide(healthexp, 'Year', 'Country', 'Spending_USD')

Group combos are unique.


Country,Canada,France,Germany,Great Britain,Japan,USA
Year,,,,,,
1970,NaN,192.143,252.311,123.993,150.437,326.961
1971,313.391,NaN,298.251,134.172,163.854,357.988
1972,NaN,NaN,337.364,NaN,185.390,397.097
1973,NaN,NaN,384.541,NaN,205.778,439.302
1974,NaN,NaN,452.744,NaN,242.018,495.114
1975,NaN,363.610,532.481,NaN,284.269,560.750
1976,543.337,NaN,591.098,NaN,303.725,638.851
1977,NaN,NaN,647.352,NaN,340.628,726.241
1978,NaN,NaN,729.457,NaN,392.577,808.884


In [76]:
tidy2wide(healthexp, 'Year', 'Country', 'Spending_USD', method='unstack')

Group combos are unique.


Country,Canada,France,Germany,Great Britain,Japan,USA
Year,,,,,,
1970,NaN,192.143,252.311,123.993,150.437,326.961
1971,313.391,NaN,298.251,134.172,163.854,357.988
1972,NaN,NaN,337.364,NaN,185.390,397.097
1973,NaN,NaN,384.541,NaN,205.778,439.302
1974,NaN,NaN,452.744,NaN,242.018,495.114
1975,NaN,363.610,532.481,NaN,284.269,560.750
1976,543.337,NaN,591.098,NaN,303.725,638.851
1977,NaN,NaN,647.352,NaN,340.628,726.241
1978,NaN,NaN,729.457,NaN,392.577,808.884
